# 03. Training & Evaluation

**Phase 4**: 모든 오염 데이터셋 × 5개 모델 학습 & 평가

**사전 조건**: `01`, `02` 노트북 실행 완료

---

## 0. 환경 설정

In [ ]:
# ============================================================
# 0-1. Google Drive 마운트 & 의존성
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

%pip install -q xgboost

import os, glob
import pandas as pd
import numpy as np
from time import time
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/capstone/dsc'
RAW_DIR = f'{BASE}/data/raw'
TRAIN_DIR = f'{BASE}/data/train_polluted'
TEST_DIR = f'{BASE}/data/test_clean'
RESULTS_DIR = f'{BASE}/results'

print('환경 설정 완료')


Mounted at /content/drive
환경 설정 완료


In [ ]:
# ============================================================
# 0-2. 데이터셋 메타데이터 & 모델 정의
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

DATASETS = {
    'TelcoCustomerChurn': {
        'target': 'Churn',
        'numerical_cols': ['tenure', 'MonthlyCharges', 'TotalCharges'],
        'categorical_cols': [
            'gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaperlessBilling', 'PaymentMethod'
        ],
    },
    'SouthGermanCredit': {
        'target': 'credit_risk',
        'numerical_cols': ['duration', 'amount', 'age'],
        'categorical_cols': [
            'status', 'credit_history', 'purpose', 'savings',
            'employment_duration', 'installment_rate', 'personal_status_sex',
            'other_debtors', 'present_residence', 'property',
            'other_installment_plans', 'housing', 'number_credits',
            'job', 'people_liable', 'telephone', 'foreign_worker'
        ],
    },
    'letter': {
        'target': 'lettr',
        'numerical_cols': [
            'x-box', 'y-box', 'width', 'high', 'onpix', 'x-bar',
            'y-bar', 'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br',
            'x-ege', 'xegvy', 'y-ege', 'yegvx'
        ],
        'categorical_cols': [],
    },
}


def get_models():
    return {
        'LogisticRegression': LogisticRegression(
            solver='lbfgs', max_iter=2000, random_state=42, n_jobs=-1
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        'XGBoost': XGBClassifier(
            n_estimators=100, random_state=42, n_jobs=-1,
            use_label_encoder=False, eval_metric='mlogloss'
        ),
        'SVC': SVC(
            kernel='linear', random_state=42, probability=True,
            class_weight='balanced',
        ),
        'MLP': MLPClassifier(
            hidden_layer_sizes=(100, 100, 100, 100, 100),
            random_state=42, max_iter=1000
        ),
    }


def preprocess(df_train, df_test, meta):
    """이미 분리된 train/test를 받아서 전처리만 수행. split 로직 없음."""
    target = meta['target']
    num_cols = [c for c in meta['numerical_cols'] if c in df_train.columns]
    cat_cols = [c for c in meta['categorical_cols'] if c in df_train.columns]
    feature_cols = num_cols + cat_cols

    X_train = df_train[feature_cols].copy()
    X_test = df_test[feature_cols].copy()

    le = LabelEncoder()
    y_train = le.fit_transform(df_train[target].astype(str))
    y_test = le.transform(df_test[target].astype(str))

    for col in num_cols:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
    for col in num_cols:
        med = X_train[col].median() if X_train[col].notna().any() else 0
        X_train[col] = X_train[col].fillna(med)
        X_test[col] = X_test[col].fillna(med)
    for col in cat_cols:
        X_train[col] = X_train[col].fillna('MISSING').astype(str)
        X_test[col] = X_test[col].fillna('MISSING').astype(str)

    transformers = []
    if num_cols:
        transformers.append(('num', StandardScaler(), num_cols))
    if cat_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols))

    preprocessor = ColumnTransformer(transformers, remainder='drop')
    X_train_t = preprocessor.fit_transform(X_train)
    X_test_t = preprocessor.transform(X_test)

    n_classes = len(le.classes_)
    return X_train_t, X_test_t, y_train, y_test, n_classes


def evaluate_model(model, X_test, y_test, n_classes):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    try:
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test)
        else:
            y_prob = model.decision_function(X_test)
        if n_classes == 2:
            if y_prob.ndim == 2:
                y_prob = y_prob[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        else:
            auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    except Exception:
        auc = np.nan
    return {'accuracy': round(acc, 4), 'f1_macro': round(f1, 4), 'auc_roc': round(auc, 4)}


print('모델 및 유틸리티 정의 완료')


모델 및 유틸리티 정의 완료


## 1. 오염 데이터 목록 스캔

In [ ]:
# ============================================================
# 1-1. 실험 목록 구성 (train_polluted 디렉토리 스캔)
# ============================================================
experiments = []

for ds_name in DATASETS:
    ds_dir = f'{TRAIN_DIR}/{ds_name}'
    if not os.path.isdir(ds_dir):
        continue
    for folder in sorted(os.listdir(ds_dir)):
        train_csv = f'{ds_dir}/{folder}/train_data.csv'
        if not os.path.isfile(train_csv):
            continue
        parts = folder.rsplit('_', 1)
        polluter_name = parts[0]
        level = int(parts[1]) / 100 if len(parts) == 2 else 0.0
        experiments.append({
            'dataset': ds_name,
            'polluter': polluter_name,
            'level': level,
            'train_path': train_csv,
        })

print(f'실험 목록: {len(experiments)}건')
print(f'모델 5개 x {len(experiments)}건 = 총 {5 * len(experiments)}회 학습 예정')


실험 목록: 87건
모델 5개 x 87건 = 총 435회 학습 예정


## 1-2. Leakage 자동 검증 (안전장치)

학습 시작 전, 모든 train 데이터에 test 행이 포함되어 있지 않은지 자동 검증합니다.
1건이라도 발견되면 에러를 발생시켜 학습을 중단합니다.

In [ ]:
# ============================================================
# 1-2. Leakage 검증 (학습 전 안전장치)
# ============================================================
# 1차: split 인덱스 disjoint (노트북 02가 저장한 train_idx, test_idx)
# 2차: row hash 비교 (자연 중복 baseline 화이트리스트 차감)
# ============================================================
import hashlib

SPLIT_META_DIR = f'{BASE}/data/split_meta'

def _row_hash(row):
    return hashlib.md5('|'.join(map(str, row.values)).encode()).hexdigest()

print('=== Leakage 검증 시작 ===')
leakage_found = False

for ds_name in DATASETS:
    test_path = f'{TEST_DIR}/{ds_name}_test.csv'
    if not os.path.isfile(test_path):
        print(f'  {ds_name}: test 파일 없음 -> SKIP')
        continue

    # --- 1차: split 인덱스 disjoint ---
    train_idx_p = f'{SPLIT_META_DIR}/{ds_name}_train_idx.npy'
    test_idx_p = f'{SPLIT_META_DIR}/{ds_name}_test_idx.npy'
    if os.path.isfile(train_idx_p) and os.path.isfile(test_idx_p):
        train_idx = set(np.load(train_idx_p).tolist())
        test_idx = set(np.load(test_idx_p).tolist())
        idx_overlap = train_idx & test_idx
        if idx_overlap:
            print(f'  {ds_name}: ❌ split 인덱스 겹침 {len(idx_overlap)}건 — 02 노트북 split 검증 실패!')
            leakage_found = True
            continue
        print(f'  {ds_name}: split disjoint ✓ (train {len(train_idx)}, test {len(test_idx)})')
    else:
        print(f'  {ds_name}: split_meta 없음 → 2차 검증으로만 확인')

    # --- 2차: row hash 기반 (split-first 가정 검증) ---
    df_test = pd.read_csv(test_path)
    test_hashes = set(df_test.apply(_row_hash, axis=1))

    baseline_path = f'{TRAIN_DIR}/{ds_name}/none_0/train_data.csv'
    df_baseline = pd.read_csv(baseline_path)
    baseline_hashes = set(df_baseline.apply(_row_hash, axis=1))
    natural_overlap = baseline_hashes & test_hashes
    if natural_overlap:
        print(f'  {ds_name}: 자연 중복 {len(natural_overlap)}건 (baseline에 이미 존재)')

    ds_experiments = [e for e in experiments if e['dataset'] == ds_name]
    new_leak_count = 0
    for exp in ds_experiments:
        df_train = pd.read_csv(exp['train_path'])
        train_hashes = set(df_train.apply(_row_hash, axis=1))
        new_overlap = (train_hashes & test_hashes) - natural_overlap
        if new_overlap:
            label = f"{ds_name}/{exp['polluter']}_{int(exp['level']*100)}%"
            print(f'    LEAKAGE: {label} → 새 겹침 {len(new_overlap)}건')
            leakage_found = True
            new_leak_count += 1

    print(f'  {ds_name}: {len(ds_experiments)}건 검증 완료 (신규 leakage {new_leak_count}건)')

if leakage_found:
    raise RuntimeError('LEAKAGE 발견! 02 노트북의 split·폴루션 단계를 확인하세요.')
print()
print('모든 실험에서 leakage 없음 확인. 학습 진행 가능.')

=== Leakage 검증 시작 ===
  TelcoCustomerChurn: split disjoint ✓ (train 5634, test 1409)
  TelcoCustomerChurn: 31건 검증 완료 (신규 leakage 0건)
  SouthGermanCredit: split disjoint ✓ (train 800, test 200)
  SouthGermanCredit: 31건 검증 완료 (신규 leakage 0건)
  letter: split disjoint ✓ (train 16000, test 4000)
  letter: 자연 중복 312건 (baseline에 이미 존재)
  letter: 25건 검증 완료 (신규 leakage 0건)

모든 실험에서 leakage 없음 확인. 학습 진행 가능.


## 2. 전체 모델 학습 & 평가

In [ ]:
# ============================================================
# 2-1. 학습 루프 (체크포인트 지원)
# ============================================================
perf_path = f'{RESULTS_DIR}/model_performance.csv'

if os.path.isfile(perf_path):
    df_perf = pd.read_csv(perf_path)
    existing_keys = set(
        df_perf.apply(lambda r: f"{r['dataset']}|{r['polluter']}|{r['level']}|{r['model']}", axis=1)
    )
    perf_rows = df_perf.to_dict('records')
    print(f'기존 결과 {len(perf_rows)}건 로드')
else:
    existing_keys = set()
    perf_rows = []

total_start = time()
error_log = []
completed = 0
skipped = 0

for i, exp in enumerate(experiments):
    ds_name = exp['dataset']
    meta = DATASETS[ds_name]

    try:
        df_train = pd.read_csv(exp['train_path'])
        df_test = pd.read_csv(f'{TEST_DIR}/{ds_name}_test.csv')
        if 'TotalCharges' in df_train.columns:
            df_train['TotalCharges'] = pd.to_numeric(df_train['TotalCharges'], errors='coerce').fillna(0)
        if 'TotalCharges' in df_test.columns:
            df_test['TotalCharges'] = pd.to_numeric(df_test['TotalCharges'], errors='coerce').fillna(0)
        X_train, X_test, y_train, y_test, n_classes = preprocess(df_train, df_test, meta)
    except Exception as e:
        label = f"{ds_name}/{exp['polluter']}_{int(exp['level']*100)}%"
        error_log.append({'label': label, 'model': 'ALL', 'error': str(e)})
        print(f'  [{i+1}/{len(experiments)}] {label} -> 전처리 실패: {e}')
        continue

    for model_name, model in get_models().items():
        key = f"{ds_name}|{exp['polluter']}|{exp['level']}|{model_name}"
        if key in existing_keys:
            skipped += 1
            continue
        try:
            t0 = time()
            model.fit(X_train, y_train)
            scores = evaluate_model(model, X_test, y_test, n_classes)
            elapsed = time() - t0
            row = {
                'dataset': ds_name,
                'polluter': exp['polluter'],
                'level': exp['level'],
                'model': model_name,
                **scores,
            }
            perf_rows.append(row)
            existing_keys.add(key)
            completed += 1
        except Exception as e:
            label = f"{ds_name}/{exp['polluter']}_{int(exp['level']*100)}%/{model_name}"
            error_log.append({'label': label, 'error': str(e)})

    if (i + 1) % 5 == 0 or i == len(experiments) - 1:
        elapsed_total = time() - total_start
        print(f'  [{i+1}/{len(experiments)}] 완료={completed}, 스킵={skipped}, 에러={len(error_log)}  ({elapsed_total:.0f}s)')

    if (i + 1) % 10 == 0:
        pd.DataFrame(perf_rows).to_csv(perf_path, index=False)

df_perf = pd.DataFrame(perf_rows)
df_perf.to_csv(perf_path, index=False)
total_elapsed = time() - total_start
print()
print('=== 학습 완료 ===')
print(f'총 {len(df_perf)}건 (신규 {completed}, 스킵 {skipped}, 에러 {len(error_log)})')
print(f'소요 시간: {total_elapsed:.0f}초 ({total_elapsed/60:.1f}분)')
print(f'저장: {perf_path}')


기존 결과 15건 로드
  [5/87] 완료=25, 스킵=0, 에러=0  (72s)
  [10/87] 완료=50, 스킵=0, 에러=0  (211s)
  [15/87] 완료=75, 스킵=0, 에러=0  (396s)
  [20/87] 완료=100, 스킵=0, 에러=0  (556s)
  [25/87] 완료=120, 스킵=5, 에러=0  (679s)
  [30/87] 완료=145, 스킵=5, 에러=0  (1981s)
  [35/87] 완료=170, 스킵=5, 에러=0  (2756s)
  [40/87] 완료=195, 스킵=5, 에러=0  (2769s)
  [45/87] 완료=220, 스킵=5, 에러=0  (2784s)
  [50/87] 완료=245, 스킵=5, 에러=0  (2796s)
  [55/87] 완료=270, 스킵=5, 에러=0  (2809s)
  [60/87] 완료=290, 스킵=10, 에러=0  (2835s)
  [65/87] 완료=315, 스킵=10, 에러=0  (3004s)
  [70/87] 완료=340, 스킵=10, 에러=0  (3576s)
  [75/87] 완료=365, 스킵=10, 에러=0  (5189s)
  [80/87] 완료=390, 스킵=10, 에러=0  (6761s)
  [85/87] 완료=410, 스킵=15, 에러=0  (7551s)


In [ ]:
# ============================================================
# 2-2. 에러 로그 확인
# ============================================================
if error_log:
    print(f'에러 {len(error_log)}건:')
    for e in error_log:
        print(f'  {e["label"]}: {e.get("error", "unknown")}')
else:
    print('에러 없음')

In [ ]:
# ============================================================
# 2-3. 결과 미리보기
# ============================================================
print(f'\n결과 shape: {df_perf.shape}')
print(f'\n데이터셋별 건수:')
print(df_perf.groupby('dataset').size())
print(f'\npolluter별 건수:')
print(df_perf.groupby('polluter').size())
print(f'\n--- 노트북 03 완료 ---')
print(f'다음: 04_scoreboard.ipynb 실행')

df_perf.head(10)

In [ ]:
# ============================================================
# 3. 실행 로그 저장
# ============================================================
from datetime import datetime

log_lines = []
log_lines.append('# 노트북 03 실행 로그: 모델 학습 & 평가')
log_lines.append('')
log_lines.append(f'- **실행 시각**: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
log_lines.append(f'- **총 실험**: {len(df_perf)}건')
log_lines.append(f'- **신규 학습**: {completed}건')
log_lines.append(f'- **스킵 (기존 결과)**: {skipped}건')
log_lines.append(f'- **에러**: {len(error_log)}건')
log_lines.append(f'- **소요 시간**: {total_elapsed:.0f}초 ({total_elapsed/60:.1f}분)')
log_lines.append('')

# 모델 설정
log_lines.append('## 1. 모델 설정')
log_lines.append('')
log_lines.append('| 모델 | 계열 | 주요 하이퍼파라미터 |')
log_lines.append('|---|---|---|')
log_lines.append('| LogisticRegression | 선형 | solver=lbfgs, max_iter=2000 |')
log_lines.append('| RandomForest | 배깅 | n_estimators=100 |')
log_lines.append('| XGBoost | 부스팅 | n_estimators=100 |')
log_lines.append('| SVC | 커널 | kernel=linear |')
log_lines.append('| MLP | 신경망 | hidden_layers=(100,100,100,100,100), max_iter=1000 |')
log_lines.append('')
log_lines.append('- 전처리: StandardScaler(수치) + OneHotEncoder(범주)')
log_lines.append('- 분할: train 80% / test 20% (stratified)')
log_lines.append('- 평가: Accuracy, F1(macro), AUC-ROC')
log_lines.append('')

# 데이터셋별 결과 요약
log_lines.append('## 2. 데이터셋별 결과 요약')
log_lines.append('')
for ds_name in df_perf['dataset'].unique():
    ds_df = df_perf[df_perf['dataset'] == ds_name]
    log_lines.append(f'### {ds_name} ({len(ds_df)}건)')
    log_lines.append('')
    log_lines.append('| 오염 유형 | 강도 | 모델 | F1(macro) | Accuracy | AUC-ROC |')
    log_lines.append('|---|---|---|---|---|---|')
    for _, row in ds_df.iterrows():
        log_lines.append(f'| {row["polluter"]} | {row["level"]} | {row["model"]} | {row["f1_macro"]} | {row["accuracy"]} | {row["auc_roc"]} |')
    log_lines.append('')

# 모델별 평균 성능
log_lines.append('## 3. 모델별 평균 F1 (전체)')
log_lines.append('')
log_lines.append('| 모델 | 평균 F1 | 최소 F1 | 최대 F1 | 건수 |')
log_lines.append('|---|---|---|---|---|')
model_stats = df_perf.groupby('model')['f1_macro'].agg(['mean', 'min', 'max', 'count'])
for model_name, stats in model_stats.iterrows():
    log_lines.append(f'| {model_name} | {stats["mean"]:.4f} | {stats["min"]:.4f} | {stats["max"]:.4f} | {int(stats["count"])} |')
log_lines.append('')

# 에러
if error_log:
    log_lines.append('## 4. 에러 목록')
    log_lines.append('')
    for e in error_log:
        err_msg = str(e.get('error', 'unknown'))
        if len(err_msg) > 200:
            err_msg = err_msg[:200] + '...'
        log_lines.append(f'- **{e.get("label", "?")}**: {err_msg}')
    log_lines.append('')

# 산출물
log_lines.append('## 5. 산출물')
log_lines.append('')
log_lines.append(f'- `model_performance.csv` — 모델 성능 {len(df_perf)}건')
log_lines.append(f'- `03_execution_log.md` — 이 로그 파일')
log_lines.append('')
log_lines.append('---')
log_lines.append('*이 로그는 노트북 03 실행 시 자동 생성됨*')

log_path = f'{RESULTS_DIR}/03_execution_log.md'
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))
print(f'실행 로그 저장: {log_path}')
